> ## ⚠️ ARCHIVED — the results in this notebook are INVALID
>
> Kept as a record of how the project started. **Do not use its numbers.**
> It implements four methodological errors that the package pipeline fixes,
> and each one flatters the result:
>
> | defect | in this notebook | why it is wrong |
> |---|---|---|
> | Alphabetical **truncation** of the universe | `sample = tickers[:SAMPLE_N]` | a 500-cap covered roughly A–POOL, not a cross-section of the index |
> | **Point-in-time** filter applied *before* time-series features | `filter_panel_to_pit(labeled, stints)` then features | rolling windows see gaps where a name left the index, so momentum and volatility are computed across holes |
> | Unlabelled rows **dropped** | `long.dropna(subset=["fwd_ret_10d"])` | removes the newest sessions — exactly the rows a live model has to rank |
> | Row-based, **unpurged** `TimeSeriesSplit` | `TimeSeriesSplit(n_splits=...)` | splits cut through a date, and no purge means the label horizon leaks across the boundary |
>
> On identical portfolio rules, the truncated universe showed **+76.6% and Sharpe 1.05** where the corrected panel shows **+22.2% and Sharpe 0.16**. Outputs have been cleared so those figures are not displayed as findings.
>
> For anything current use the package: `train-sp500`, `backtest-sp500`, `predict-sp500`, or the `stock_predictor` API. See the README's Limitations section for what is measured and how.


# S&P 500 Weekly Move Predictor

**Goal:** Predict which S&P 500 stocks will gain ≥5% over the next 10 trading days.

**Pipeline:**
1. Download tickers & prices
2. Build binary label (fwd return ≥ 5%)
3. Feature engineering (price, volume, sector-relative)
4. Optional **Optuna** hyperparameter search with `TimeSeriesSplit` on the train panel
5. LightGBM model with time-based split
6. Evaluation: PR-AUC and Precision@k
7. **Walk-forward** validation (expanding monthly window)
8. Feature importance comparison

Run with: `uv run jupyter notebook` or `uv run jupyter lab` from the project root.

## 0. Imports & settings

In [ ]:
from __future__ import annotations

import io
import time
import urllib.request

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit

import optuna

from pathlib import Path
import sys

for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "pyproject.toml").exists():
        sys.path.insert(0, str(_root.resolve()))
        break

from sp500_pit import filter_panel_to_pit, load_sp500_stints, tickers_overlapping_window
from calendar_features import CALENDAR_FEATURE_COLS, add_calendar_features

pd.set_option("display.max_columns", 20)
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

# ── Global constants ──────────────────────────────────────────────────────
HORIZON    = 10      # trading sessions forward
THRESHOLD  = 0.05    # +5% gain to be labelled positive
START      = "2018-01-01"
END        = None    # through today
TRAIN_END  = "2022-12-31"
TEST_START = "2023-01-01"
SAMPLE_N   = 500  # set to e.g. 50 while developing


## 1. S&P 500 universe (point-in-time)

Universe and row filtering use **[fja05680/sp500](https://github.com/fja05680/sp500)** `sp500_ticker_start_end.csv`: each row is a membership stint `[start_date, end_date)` (blank `end_date` = still in the index per that file). This is **community-maintained, not official S&P** — cross-check a vendor dataset (e.g. Compustat via WRDS) for production.

**Wikipedia** is fetched only for **GICS sector** on symbols in the current table; tickers not listed there (often delisted) get `sector = NaN`.

With `SAMPLE_N` below the full union size, we take the first *N* tickers after sorting (alphabetical bias).

In [ ]:
SP500_STINTS_URL = (
    "https://raw.githubusercontent.com/fja05680/sp500/master/sp500_ticker_start_end.csv"
)
stints = load_sp500_stints(SP500_STINTS_URL)
tickers = tickers_overlapping_window(stints, START, END)
print(f"Tickers (union of stints overlapping [START, END]): {len(tickers)}")

# Wikipedia: sector labels for current S&P names only (not used for universe).
SP500_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
_WIKI_HEADERS = {
    "User-Agent": "stock-predictor/0.1 (Python/pandas; educational research)",
}
req = urllib.request.Request(SP500_URL, headers=_WIKI_HEADERS)
with urllib.request.urlopen(req, timeout=30) as resp:
    html = resp.read().decode("utf-8")
tables = pd.read_html(io.StringIO(html))
wiki_constituents = tables[0]
wiki_constituents.head()

## 2. Download daily OHLCV prices

We download with `auto_adjust=True` so Close is already split/dividend-adjusted.

In [ ]:
sample = tickers[:SAMPLE_N]
data = yf.download(
    sample,
    start=START,
    end=END,
    group_by="ticker",
    threads=True,
    auto_adjust=True,
    progress=False,
)

def wide_field(raw: pd.DataFrame, field: str) -> pd.DataFrame:
    """Extract a single field (Close, Volume, …) from a multi-ticker download."""
    if isinstance(raw.columns, pd.MultiIndex):
        return raw.xs(field, axis=1, level=-1).sort_index()
    return raw[[field]].sort_index()

adj_close = wide_field(data, "Close")
volume    = wide_field(data, "Volume")

print(f"adj_close shape: {adj_close.shape}")
adj_close.tail()

## 3. Forward return & binary label

For each (date, ticker): close-to-close return from `t` to `t + HORIZON`.  
`target_5pct = 1` if that forward return ≥ THRESHOLD.

In [ ]:
# ── Adjusted close: long format ───────────────────────────────────────────
long = (
    adj_close
    .stack(future_stack=True)
    .rename("adj_close")
    .reset_index()
)
long.columns = ["date", "ticker", "adj_close"]
long = long.sort_values(["ticker", "date"])

# ── Forward return & label ────────────────────────────────────────────────
long["fwd_ret_10d"] = long.groupby("ticker", group_keys=False)["adj_close"].transform(
    lambda s: s.shift(-HORIZON) / s - 1.0
)
long["target_5pct"] = (long["fwd_ret_10d"] >= THRESHOLD).astype("int8")

labeled = long.dropna(subset=["fwd_ret_10d"])
labeled = filter_panel_to_pit(labeled, stints)

pos_rate = labeled["target_5pct"].mean()
print(f"Positive rate (>= {THRESHOLD:.0%} in {HORIZON} sessions): {pos_rate:.4%}")
labeled.head(10)

### Label distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
labeled["fwd_ret_10d"].clip(-0.5, 0.5).hist(
    bins=80, ax=ax[0], color="steelblue", edgecolor="white"
)
ax[0].set_title("Forward 10d return (clipped ±50%)")
ax[0].set_xlabel("return")
labeled["target_5pct"].value_counts().sort_index().plot(
    kind="bar", ax=ax[1], color=["#ccc", "#2ca02c"]
)
ax[1].set_title("Label distribution")
ax[1].set_xticklabels(["0", "1"], rotation=0)
plt.tight_layout()
plt.show()

## 4. Feature engineering

All row-level features are aligned so we do **not** use future stock prices or future earnings that were not yet known at `t`. Caveats:
- **Macro** (4d): VIX & yields are merged on **calendar date** (same close timestamp assumption as the stock bar).
- **Calendar** (4e): month (sin/cos), quarter-end flags, and **days to next FOMC** from a static statement-date list in `calendar_features.py` (extend dates for years beyond the list; distant tail is capped).
- **Earnings** (4f): **days since last reported quarter** only (no “days to next” using future dates).
- **Short interest** is **omitted** until point-in-time (e.g. FINRA) data is wired in — Yahoo’s snapshot fields would leak look-ahead if used as fake history.

### 4a. Price-based features

In [ ]:
def add_price_features(df: pd.DataFrame) -> pd.DataFrame:
    g_price = df.groupby("ticker", group_keys=False)["adj_close"]

    # Past returns
    for lag in [1, 5, 10, 21]:
        df[f"ret_{lag}d"] = g_price.transform(lambda s: s.pct_change(lag))

    # Momentum: 21-day minus 5-day (avoids short-term reversal contamination)
    df["momentum"] = df["ret_21d"] - df["ret_5d"]

    # Rolling volatility
    daily_ret = g_price.transform(lambda s: s.pct_change())
    df["vol_10d"] = daily_ret.groupby(df["ticker"]).transform(
        lambda s: s.rolling(10).std()
    )
    df["vol_21d"] = daily_ret.groupby(df["ticker"]).transform(
        lambda s: s.rolling(21).std()
    )

    # RSI (14-day)
    def rsi(s: pd.Series, window: int = 14) -> pd.Series:
        delta = s.diff()
        gain  = delta.clip(lower=0).rolling(window).mean()
        loss  = (-delta.clip(upper=0)).rolling(window).mean()
        rs    = gain / loss.replace(0, np.nan)
        return 100 - 100 / (1 + rs)

    df["rsi_14"] = g_price.transform(rsi)

    # Price vs rolling mean
    df["price_vs_ma20"] = g_price.transform(lambda s: s / s.rolling(20).mean() - 1)
    df["price_vs_ma50"] = g_price.transform(lambda s: s / s.rolling(50).mean() - 1)

    # Bollinger Band position (0 = lower band, 1 = upper band)
    def bb_position(s: pd.Series, window: int = 20) -> pd.Series:
        ma    = s.rolling(window).mean()
        std   = s.rolling(window).std()
        upper = ma + 2 * std
        lower = ma - 2 * std
        return (s - lower) / (upper - lower + 1e-9)

    df["bb_pos"] = g_price.transform(bb_position)

    return df


features = add_price_features(labeled.copy())

PRICE_FEATURE_COLS = [
    "ret_1d", "ret_5d", "ret_10d", "ret_21d",
    "momentum", "vol_10d", "vol_21d",
    "rsi_14", "price_vs_ma20", "price_vs_ma50", "bb_pos",
]
print(f"Panel shape after price features: {features.shape}")
features[PRICE_FEATURE_COLS].describe()

### 4b. Volume features

In [ ]:
# Merge volume into the panel
vol_long = (
    volume
    .stack(future_stack=True)
    .rename("volume")
    .reset_index()
)
vol_long.columns = ["date", "ticker", "volume"]
features = features.merge(vol_long, on=["date", "ticker"], how="left")


def add_volume_features(df: pd.DataFrame) -> pd.DataFrame:
    g_vol = df.groupby("ticker", group_keys=False)["volume"]

    # Z-score: how unusual is today's volume vs the past 21 days?
    vol_ma21  = g_vol.transform(lambda s: s.rolling(21).mean())
    vol_std21 = g_vol.transform(lambda s: s.rolling(21).std())
    df["volume_zscore"] = (df["volume"] - vol_ma21) / (vol_std21 + 1e-9)

    # Short-term volume spike vs 5-day average
    vol_ma5 = g_vol.transform(lambda s: s.rolling(5).mean())
    df["volume_ratio_5d"] = df["volume"] / (vol_ma5 + 1e-9)

    # Volume trend: is volume expanding or contracting?
    df["volume_trend_10d"] = g_vol.transform(
        lambda s: s.rolling(10).mean() / s.rolling(21).mean() - 1
    )

    # Price-volume divergence: strong price move on weak volume = suspect
    price_ret_5d = df.groupby("ticker", group_keys=False)["adj_close"].transform(
        lambda s: s.pct_change(5)
    )
    vol_chg_5d = g_vol.transform(lambda s: s.pct_change(5))
    df["price_vol_divergence"] = price_ret_5d - vol_chg_5d

    return df


features = add_volume_features(features)

VOLUME_FEATURE_COLS = [
    "volume_zscore",
    "volume_ratio_5d",
    "volume_trend_10d",
    "price_vol_divergence",
]
print(f"NaNs in volume features:\n{features[VOLUME_FEATURE_COLS].isna().sum()}")
features[VOLUME_FEATURE_COLS].describe()

### 4c. Sector features

Two things:
- **Sector identity** as a categorical feature (LightGBM handles natively)
- **Sector-relative momentum/vol** — how is this stock performing vs its peers?

In [ ]:
sector_map = (
    wiki_constituents[["Symbol", "GICS Sector"]]
    .rename(columns={"Symbol": "ticker", "GICS Sector": "sector"})
    .assign(ticker=lambda df: df["ticker"].str.replace(".", "-", regex=False))
)
features = features.merge(sector_map, on="ticker", how="left")

print(f"Sectors: {features['sector'].nunique()}")
print(features["sector"].value_counts())


def add_sector_relative_features(df: pd.DataFrame) -> pd.DataFrame:
    # Sector-median return: how does this stock rank within its sector?
    for window in [5, 21]:
        col = f"ret_{window}d"
        sector_median = df.groupby(["date", "sector"])[col].transform("median")
        df[f"{col}_vs_sector"] = df[col] - sector_median

    # Sector-relative volatility
    sector_vol_median = df.groupby(["date", "sector"])["vol_10d"].transform("median")
    df["vol_vs_sector"] = df["vol_10d"] - sector_vol_median

    return df


features = add_sector_relative_features(features)
features["sector"] = features["sector"].astype("category")

SECTOR_FEATURE_COLS = [
    "sector",
    "ret_5d_vs_sector",
    "ret_21d_vs_sector",
    "vol_vs_sector",
]
features[SECTOR_FEATURE_COLS].head(10)

### 4d. Macro (VIX & yield curve)

Daily series from Yahoo: **^VIX** (fear gauge), **^TNX** (10Y Treasury yield, %), **^IRX** (13-week T-bill yield, %).  
**`yield_curve_spread`** = TNX − IRX (simple slope proxy; not exactly 10Y−2Y).  
`vix_ret_5d` is the 5-session **percentage** change in VIX (same calendar merge as the stock panel).

### 4e. Calendar features

**Month** as sin/cos (1–12); **quarter-end**: indicator for Mar/Jun/Sep/Dec and **calendar days** to quarter-end; **FOMC**: calendar days until the next listed statement date (Fed schedule in `calendar_features.py`; values beyond the last date are capped). Implemented in the next code cell after the macro merge.

In [ ]:
MACRO_YF = ["^VIX", "^TNX", "^IRX"]

macro_dl = yf.download(
    MACRO_YF,
    start=START,
    end=END,
    group_by="ticker",
    threads=True,
    auto_adjust=False,
    progress=False,
)

def _close_wide_macro(raw: pd.DataFrame) -> pd.DataFrame:
    """Macro indices use columns (Price, Ticker); stocks use (Ticker, Price)."""
    if not isinstance(raw.columns, pd.MultiIndex):
        return raw[["Close"]].sort_index()
    if raw.columns.names[0] == "Price":
        return raw.xs("Close", level=0, axis=1).sort_index()
    return raw.xs("Close", level=-1, axis=1).sort_index()

try:
    mw = _close_wide_macro(macro_dl)
    mdf = mw.rename(columns={"^VIX": "vix", "^TNX": "tnx_yield", "^IRX": "irx_yield"}).reset_index()
    dcol = mdf.columns[0]
    mdf = mdf.rename(columns={dcol: "date"})
    mdf["date"] = pd.to_datetime(mdf["date"]).dt.normalize()
    mdf["yield_curve_spread"] = mdf["tnx_yield"] - mdf["irx_yield"]
    mdf["vix_ret_5d"] = mdf["vix"].pct_change(5)
    macro_panel = mdf[["date", "vix", "vix_ret_5d", "tnx_yield", "yield_curve_spread"]].copy()
except Exception as exc:
    print("Macro download failed, using empty macro features:", exc)
    macro_panel = pd.DataFrame(
        columns=["date", "vix", "vix_ret_5d", "tnx_yield", "yield_curve_spread"]
    )

features["date"] = pd.to_datetime(features["date"]).dt.normalize()
features = features.merge(macro_panel, on="date", how="left")

MACRO_FEATURE_COLS = ["vix", "vix_ret_5d", "tnx_yield", "yield_curve_spread"]
print("Macro NaNs:\n", features[MACRO_FEATURE_COLS].isna().sum())
features[MACRO_FEATURE_COLS].describe()

features = add_calendar_features(features)
print("Calendar features:\n", features[CALENDAR_FEATURE_COLS].describe())


### 4f. Earnings proximity

For each row date, **`days_since_last_earnings`** = trading days since the **last reported** earnings date (from `yfinance.get_earnings_dates`), using only dates **strictly before** the row’s `date` — no lookahead to future reports.

Many tickers will have **NaN** early in history or if Yahoo has no table; LightGBM can split on missingness.

Uses parallel requests with a small throttle; increase `SAMPLE_N` cautiously.

In [ ]:
def _fetch_earnings_index(ticker: str) -> pd.DatetimeIndex:
    try:
        time.sleep(0.05)
        ed = yf.Ticker(ticker).get_earnings_dates(limit=28)
        if ed is None or len(ed) == 0:
            return pd.DatetimeIndex([])
        return pd.to_datetime(ed.index, utc=False).normalize().sort_values().unique()
    except Exception:
        return pd.DatetimeIndex([])


def build_earnings_map(tickers: list[str], max_workers: int = 8) -> dict[str, np.ndarray]:
    out: dict[str, np.ndarray] = {}
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = {ex.submit(_fetch_earnings_index, t): t for t in tickers}
        for fut in as_completed(futs):
            t = futs[fut]
            try:
                idx = fut.result()
            except Exception:
                idx = pd.DatetimeIndex([])
            out[t] = idx.values.astype("datetime64[ns]")
    return out


def add_days_since_last_earnings(df: pd.DataFrame, earn_map: dict[str, np.ndarray]) -> pd.DataFrame:
    pieces: list[pd.DataFrame] = []
    for t, g in df.groupby("ticker", sort=False):
        g = g.sort_values("date").copy()
        earn = earn_map.get(t)
        if earn is None or len(earn) == 0:
            g["days_since_last_earnings"] = np.nan
            pieces.append(g)
            continue
        d_ns = g["date"].values.astype("datetime64[ns]")
        pos = np.searchsorted(earn, d_ns, side="right") - 1
        days = np.full(len(g), np.nan, dtype=float)
        ok = pos >= 0
        days[ok] = (d_ns[ok] - earn[pos[ok]]) / np.timedelta64(1, "D")
        g["days_since_last_earnings"] = days
        pieces.append(g)
    return pd.concat(pieces, ignore_index=True)


earn_map = build_earnings_map(sorted(features["ticker"].unique()))
features = add_days_since_last_earnings(features, earn_map)

EARNINGS_FEATURE_COLS = ["days_since_last_earnings"]
print("Earnings feature NaNs:", features["days_since_last_earnings"].isna().mean())
features[EARNINGS_FEATURE_COLS].describe()


## 5. Train / test split (time-based)

**Never use random splits for time series** — it leaks future information into training.  
We use a hard cutoff: train on 2018–2022, test on 2023+.

In [ ]:
FEATURE_COLS = (
    PRICE_FEATURE_COLS
    + VOLUME_FEATURE_COLS
    + [
        "ret_5d_vs_sector",
        "ret_21d_vs_sector",
        "vol_vs_sector",
        "sector",
    ]
    + MACRO_FEATURE_COLS
    + CALENDAR_FEATURE_COLS
    + EARNINGS_FEATURE_COLS
)

features_clean = features.dropna(subset=FEATURE_COLS + ["target_5pct"])

train = features_clean[features_clean["date"] <= TRAIN_END]
test  = features_clean[features_clean["date"] >= TEST_START]

X_train, y_train = train[FEATURE_COLS], train["target_5pct"]
X_test,  y_test  = test[FEATURE_COLS],  test["target_5pct"]

print(f"Train: {X_train.shape} | Positive rate: {y_train.mean():.4%}")
print(f"Test:  {X_test.shape}  | Positive rate: {y_test.mean():.4%}")


## 5b. Hyperparameter tuning (Optuna + TimeSeriesSplit)

We tune on **training rows only**, sorted by `date` then `ticker`, so each CV fold trains on an **earlier contiguous index range** and validates on a **later** one — consistent with `TimeSeriesSplit` (no random shuffle).

**Objective:** mean validation **PR-AUC** across folds — better than accuracy for rare positives.

Set `RUN_OPTUNA = True` to search (can take several minutes). Set `False` to skip; §6 then uses the manual defaults below. `OPTUNA_BEST_LGBM` is always defined (empty dict if skipped).


In [ ]:
RUN_OPTUNA = True  # True to run Optuna search
N_OPTUNA_TRIALS = 40
TS_CV_SPLITS = 5
OPTUNA_SEED = 42

train_sorted = train.sort_values(["date", "ticker"]).reset_index(drop=True)
X_cv = train_sorted[FEATURE_COLS]
y_cv = train_sorted["target_5pct"].to_numpy()

neg_cv, pos_cv = int((y_cv == 0).sum()), int((y_cv == 1).sum())
spw_cv = neg_cv / pos_cv

tsc = TimeSeriesSplit(n_splits=TS_CV_SPLITS)

# Always defined so §6 can run even if this cell was skipped
OPTUNA_BEST_LGBM: dict = {}


def make_objective(X: pd.DataFrame, y: np.ndarray, spw: float, tsc, seed: int):
    def objective(trial: optuna.Trial) -> float:
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 31, 511),
            "max_depth": trial.suggest_int("max_depth", 4, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 200),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "subsample_freq": trial.suggest_int("subsample_freq", 1, 7),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        }
        fold_scores: list[float] = []
        for train_idx, val_idx in tsc.split(X):
            X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_va = y[train_idx], y[val_idx]
            clf = lgb.LGBMClassifier(
                **params,
                scale_pos_weight=spw,
                metric="average_precision",
                eval_metric="average_precision",
                random_state=seed,
                n_jobs=-1,
                verbosity=-1,
            )
            clf.fit(
                X_tr,
                y_tr,
                eval_set=[(X_va, y_va)],
                callbacks=[
                    lgb.early_stopping(80, verbose=False, first_metric_only=True),
                    lgb.log_evaluation(0),
                ],
            )
            fold_scores.append(
                average_precision_score(y_va, clf.predict_proba(X_va)[:, 1])
            )
        return float(np.mean(fold_scores))

    return objective


if RUN_OPTUNA:
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=OPTUNA_SEED),
    )
    study.optimize(
        make_objective(X_cv, y_cv, spw_cv, tsc, OPTUNA_SEED),
        n_trials=N_OPTUNA_TRIALS,
        show_progress_bar=True,
    )
    OPTUNA_BEST_LGBM = dict(study.best_params)
    print(f"Best CV mean PR-AUC: {study.best_value:.4f}")
    print("Best params:", OPTUNA_BEST_LGBM)
else:
    print(
        "RUN_OPTUNA is False — skipping search. "
        "§6 uses manual defaults unless you set RUN_OPTUNA = True and re-run this cell."
    )


## 6. LightGBM model

`scale_pos_weight` compensates for class imbalance by upweighting the positive class.

Training uses **`metric="average_precision"`** and **`eval_metric="average_precision"`** so **`early_stopping` monitors PR-AUC** on the eval set (with `eval_metric` alone, LightGBM would still stop on `binary_logloss`).

In [ ]:
if "OPTUNA_BEST_LGBM" not in globals():
    OPTUNA_BEST_LGBM = {}

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
spw = neg / pos
print(f"scale_pos_weight: {spw:.1f}  ({neg:,} negatives / {pos:,} positives)")

manual_params = {
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 63,
    "max_depth": -1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
}
if OPTUNA_BEST_LGBM:
    manual_params.update(OPTUNA_BEST_LGBM)
    print("Merged Optuna best params into model:", OPTUNA_BEST_LGBM)

model = lgb.LGBMClassifier(
    **manual_params,
    scale_pos_weight=spw,
    metric="average_precision",
    eval_metric="average_precision",
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[
        lgb.early_stopping(50, verbose=False, first_metric_only=True),
        lgb.log_evaluation(100),
    ],
)
print(f"\nBest iteration: {model.best_iteration_}")



## 7. Evaluation

With heavy class imbalance, **accuracy is meaningless**. We care about:

| Metric | Why |
|---|---|
| **PR-AUC** | Overall ranking quality for the positive class. Baseline = positive rate. |
| **ROC-AUC** | Separation quality. Baseline = 0.5. |
| **Precision@k** | Of the top-k predictions each week, how many actually hit +5%? |


In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]

pr_auc  = average_precision_score(y_test, y_prob)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"PR-AUC  : {pr_auc:.4f}  (baseline = {y_test.mean():.4f})")
print(f"ROC-AUC : {roc_auc:.4f}  (baseline = 0.5000)")

In [ ]:
def precision_at_k(y_true: pd.Series, y_scores: np.ndarray, k: int) -> float:
    """Of the top-k ranked predictions, what fraction are truly positive?"""
    if len(y_true) < k:
        return np.nan
    top_k_idx = np.argsort(y_scores)[-k:]
    return float(y_true.iloc[top_k_idx].mean())


test_scored = test.copy()
test_scored["prob"] = y_prob

# Simulate picking the top-10 candidates every week
weekly_precision = (
    test_scored
    .assign(week=lambda df: pd.to_datetime(df["date"]).dt.to_period("W"))
    .groupby("week")
    .apply(
        lambda g: precision_at_k(g["target_5pct"], g["prob"].values, k=10),
        include_groups=False,
    )
)

print(f"Mean weekly Precision@10 : {weekly_precision.mean():.4f}")
print(f"Baseline (random pick)   : {y_test.mean():.4f}")

fig, ax = plt.subplots(figsize=(13, 3))
weekly_precision.plot(ax=ax, title="Weekly Precision@10 on Test Set")
ax.axhline(y_test.mean(), color="red", linestyle="--", label="baseline (pos. rate)")
ax.set_ylabel("Precision@10")
ax.legend()
plt.tight_layout()
plt.show()

## 7b. Walk-forward validation (expanding window)

Simulate **monthly re-deployment**: for each calendar month in the test era, train on **all rows with `date` strictly before that month’s start**, then score rows **in that month only**. The training set **grows** each step (realistic backlog / retrain schedule).

- **No peeking**: test month never appears in training.
- **Early stopping**: last `INNER_VAL_FRAC` of **unique training dates** held out chronologically inside the training window; model refits on the **full** training window with `n_estimators = best_iteration + 1`.
- Requires **`manual_params`** from §6 (run §6 at least once), or edit `_wf_base` below.

Adjust `WF_TEST_START`, `WF_INNER_VAL_FRAC`, `WF_MIN_TRAIN_ROWS` as needed.

In [ ]:
# ── Walk-forward config ─────────────────────────────────────────────────────
WF_TEST_START = TEST_START  # first month to evaluate (must match string used in §5)
WF_INNER_VAL_FRAC = 0.05    # fraction of unique train dates for inner early-stopping val
WF_MIN_TRAIN_ROWS = 5_000
WF_TOP_K = 10
WF_RANDOM_STATE = 42

from IPython.display import display

if "manual_params" not in globals():
    raise RuntimeError("Run §6 first so `manual_params` exists, or define _wf_base below.")

_wf_base = manual_params.copy()


def _inner_train_val_split(train_df: pd.DataFrame, date_col: str, val_frac: float) -> tuple[pd.DataFrame, pd.DataFrame]:
    udates = np.sort(train_df[date_col].unique())
    if len(udates) < 3:
        return train_df.iloc[:0], train_df.iloc[:0]
    k = max(1, int(len(udates) * val_frac))
    val_dates = set(udates[-k:])
    tr = train_df[~train_df[date_col].isin(val_dates)]
    va = train_df[train_df[date_col].isin(val_dates)]
    return tr, va


def monthly_walk_forward(
    df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
    date_col: str,
    test_start: str,
    lgb_fixed: dict,
    *,
    inner_val_frac: float,
    min_train_rows: int,
    top_k: int,
    random_state: int,
) -> pd.DataFrame:
    """Expanding window: for each month >= test_start, train on date < month_start, test in [month_start, month_end]."""
    d = df.copy()
    d[date_col] = pd.to_datetime(d[date_col])
    first_p = pd.Timestamp(test_start).to_period("M")
    last_p = d[date_col].max().to_period("M")
    periods = pd.period_range(first_p, last_p, freq="M")

    # Avoid **lgb_fixed + n_estimators=... (duplicate keyword in one call)
    lgb_core = {k: v for k, v in lgb_fixed.items() if k != "n_estimators"}
    n_est_user = int(lgb_fixed.get("n_estimators", 500))

    records: list[dict] = []
    for p in periods:
        m_start = pd.Timestamp(year=p.year, month=p.month, day=1)
        m_end = m_start + pd.offsets.MonthEnd(0)

        train_mask = d[date_col] < m_start
        test_mask = (d[date_col] >= m_start) & (d[date_col] <= m_end)
        train_df = d.loc[train_mask]
        test_df = d.loc[test_mask]

        if len(train_df) < min_train_rows or len(test_df) == 0:
            continue

        tr_in, va_in = _inner_train_val_split(train_df, date_col, inner_val_frac)
        if len(tr_in) == 0 or len(va_in) == 0:
            continue

        y_tr = tr_in[target_col]
        neg, pos = int((y_tr == 0).sum()), int((y_tr == 1).sum())
        if neg == 0 or pos == 0:
            continue
        spw = neg / pos

        clf = lgb.LGBMClassifier(
            **lgb_core,
            n_estimators=min(2000, max(500, n_est_user * 4)),
            scale_pos_weight=spw,
            metric="average_precision",
            eval_metric="average_precision",
            random_state=random_state,
            n_jobs=-1,
            verbosity=-1,
        )
        clf.fit(
            tr_in[feature_cols],
            y_tr,
            eval_set=[(va_in[feature_cols], va_in[target_col])],
            callbacks=[
                lgb.early_stopping(80, verbose=False, first_metric_only=True),
                lgb.log_evaluation(0),
            ],
        )
        bi = clf.best_iteration_
        if bi is None:
            bi = n_est_user - 1
        n_trees = int(bi) + 1

        neg_f, pos_f = int((train_df[target_col] == 0).sum()), int((train_df[target_col] == 1).sum())
        if neg_f == 0 or pos_f == 0:
            continue
        spw_f = neg_f / pos_f

        final = lgb.LGBMClassifier(
            **lgb_core,
            n_estimators=n_trees,
            scale_pos_weight=spw_f,
            metric="average_precision",
            eval_metric="average_precision",
            random_state=random_state,
            n_jobs=-1,
            verbosity=-1,
        )
        final.fit(train_df[feature_cols], train_df[target_col])

        y_test = test_df[target_col]
        prob = final.predict_proba(test_df[feature_cols])[:, 1]

        pr = average_precision_score(y_test, prob)
        try:
            roc = roc_auc_score(y_test, prob)
        except ValueError:
            roc = np.nan

        scored = test_df.assign(prob=prob)
        weekly_p = (
            scored.assign(week=lambda x: x[date_col].dt.to_period("W"))
            .groupby("week", observed=True)
            .apply(
                lambda g: precision_at_k(g[target_col], g["prob"].values, k=top_k),
                include_groups=False,
            )
        )
        w_mean = float(np.nanmean(weekly_p.values)) if len(weekly_p) else np.nan

        records.append(
            {
                "month": str(p),
                "train_end": (m_start - pd.Timedelta(days=1)).date(),
                "n_train": len(train_df),
                "n_test": len(test_df),
                "pr_auc": pr,
                "roc_auc": roc,
                "mean_weekly_precision_at_k": w_mean,
                "pos_rate_test": float(y_test.mean()),
                "n_trees": n_trees,
            }
        )

    return pd.DataFrame.from_records(records)


wf_results = monthly_walk_forward(
    features_clean,
    FEATURE_COLS,
    "target_5pct",
    "date",
    WF_TEST_START,
    _wf_base,
    inner_val_frac=WF_INNER_VAL_FRAC,
    min_train_rows=WF_MIN_TRAIN_ROWS,
    top_k=WF_TOP_K,
    random_state=WF_RANDOM_STATE,
)

display(wf_results)

if len(wf_results) == 0:
    print(
        "No walk-forward folds produced — try lowering WF_MIN_TRAIN_ROWS or check TEST_START / data coverage."
    )
else:
    print(
        f"Walk-forward months: {len(wf_results)} | "
        f"Mean PR-AUC: {wf_results['pr_auc'].mean():.4f} | "
        f"Mean weekly Precision@{WF_TOP_K}: {wf_results['mean_weekly_precision_at_k'].mean():.4f}"
    )

    fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
    wf_results.set_index("month")["pr_auc"].astype(float).plot(
        ax=axes[0], marker="o", title="Walk-forward PR-AUC (by test month)"
    )
    axes[0].set_ylabel("PR-AUC")
    wf_results.set_index("month")["mean_weekly_precision_at_k"].astype(float).plot(
        ax=axes[1],
        marker="o",
        color="darkgreen",
        title=f"Walk-forward mean weekly Precision@{WF_TOP_K}",
    )
    axes[1].set_ylabel(f"Precision@{WF_TOP_K}")
    axes[1].tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()


## 8. Feature importance

In [ ]:
importance = (
    pd.Series(model.feature_importances_, index=FEATURE_COLS)
    .sort_values(ascending=True)
)

importance.plot(
    kind="barh",
    figsize=(9, 7),
    title="LightGBM Feature Importance (split count)",
    color="steelblue",
)
plt.tight_layout()
plt.show()

## 9. Weekly top picks (inference)

Given a date, rank all stocks by predicted probability and return the top candidates.

In [ ]:
def get_top_picks(
    df: pd.DataFrame,
    model: lgb.LGBMClassifier,
    date: str,
    feature_cols: list[str],
    top_n: int = 10,
) -> pd.DataFrame:
    """
    Return top_n stocks ranked by predicted probability for a given date.
    """
    snapshot = df[df["date"] == pd.Timestamp(date)].copy()
    if snapshot.empty:
        raise ValueError(f"No data found for date {date}")

    snapshot = snapshot.dropna(subset=feature_cols)
    snapshot["prob"] = model.predict_proba(snapshot[feature_cols])[:, 1]

    cols = ["ticker", "sector", "adj_close", "prob"] + [
        c for c in ["ret_5d", "ret_21d", "volume_zscore"] if c in snapshot.columns
    ]
    return (
        snapshot[cols]
        .sort_values("prob", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )


# Example: most recent date in the test set
last_date = test["date"].max()
print(f"Top 10 picks as of {last_date.date()}:")
get_top_picks(features_clean, model, last_date, FEATURE_COLS, top_n=10)

## Next steps

| Step | Description |
|---|---|
| **Hyperparameter tuning** | Done in §5b — extend search space or add nested CV |
| **Walk-forward validation** | §7b — expanding monthly window; tune `WF_*` constants |
| **More features** | §4d–4f; add PIT short interest, options IV, credit spreads when available |
| **Survivorship bias** | §1 uses PIT stints (fja05680); upgrade to vendor data for production |
| **Backtesting** | Use `backtrader` or `vectorbt` to simulate a real trading strategy with costs |
| **Calibration** | Use `CalibratedClassifierCV` so predicted probabilities are reliable |
